# Risk Parity Portfolio Construction

### Wealth Management Portfolio Optimization — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), introductory optimisation concepts, and basic understanding of wealth management and portfolio optimisation terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the optimisation problem and how it maps to a real wealth management and portfolio optimisation decision.
- Implement the optimisation workflow and interpret the solution in portfolio or business terms.
- Assess the sensitivity of the optimal solution to input assumptions and identify when the model may mislead.

**Estimated time:** 35–45 minutes

**Collection context:** Portfolio optimisation, asset allocation, risk management, and investment strategy for wealth management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** In a traditional 60% Stock / 40% Bond portfolio, Stocks actually account for about 90% of the total risk because they are so much more volatile. Risk Parity aims to balance the *risk* (volatility) contribution equally, so no single asset class dominates the portfolio's behavior.

**Input data used:** Asset volatilities and correlations.

**What we calculate:** Portfolio weights that lead to 'Equal Risk Contribution' (ERC).

**Math method used:** Constrained Optimization (minimizing the variance of risk contributions).

**Learning goal:** Understand that dollar diversification is not the same as risk diversification.

**Primary stakeholders:** wealth advisors, portfolio managers, investment committee members, and financial planners.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Define Asset Risk Profile

We use 4 assets with very different volatility levels.

In [ ]:
assets = ['Equities', 'Long Bonds', 'Commodities', 'T-Bills']
volatilities = np.array([0.18, 0.12, 0.25, 0.02])
corrs = np.array([
    [1.0, -0.2, 0.4, 0.0],
    [-0.2, 1.0, -0.1, 0.0],
    [0.4, -0.1, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
])

cov = np.outer(volatilities, volatilities) * corrs

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

We need a function that tells us how much risk (volatility) each asset is 'responsible' for.

In [ ]:
def risk_contribution(w, cov):
    w = np.array(w)
    port_vol = np.sqrt(w.T @ cov @ w)
    # Marginal Risk Contribution
    mrc = (cov @ w) / port_vol
    # Component Risk Contribution
    rc = w * mrc
    return rc

def risk_budget_objective(w, cov):
    # We want each RC to be 1/n_assets of the total
    n = len(w)
    rc = risk_contribution(w, cov)
    target_rc = np.sum(rc) / n
    # Minimize the squared difference between actual RC and target RC
    return np.sum((rc - target_rc)**2)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: equal-weight (1/N) portfolio ---
equal_weights = np.array([1 / n_assets] * n_assets)
eq_ret, eq_vol, eq_sharpe = portfolio_stats(equal_weights, df_returns)
print(f"Equal-weight baseline  Return: {eq_ret:.2%}  Vol: {eq_vol:.2%}  Sharpe: {eq_sharpe:.2f}")
print("Optimisation should deliver a better risk-adjusted return.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
n = len(assets)
init_w = np.array([1/n] * n)
bounds = tuple((0, 1) for _ in range(n))
cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})

res = minimize(risk_budget_objective, init_w, args=(cov,), 
               method='SLSQP', bounds=bounds, constraints=cons)

w_rp = res.x
rc_rp = risk_contribution(w_rp, cov)

print("Risk Parity Weights:")
for a, w in zip(assets, w_rp):
    print(f"{a}: {w:.2%}")

In [ ]:
rc_equal_dollar = risk_contribution(init_w, cov)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

pd.Series(rc_equal_dollar, index=assets).plot(kind='pie', ax=ax[0], autopct='%1.1f%%')
ax[0].set_title('Risk Contribution: Equal Dollar Portfolio\n(Equities dominate risk!)')

pd.Series(rc_rp, index=assets).plot(kind='pie', ax=ax[1], autopct='%1.1f%%')
ax[1].set_title('Risk Contribution: Risk Parity Portfolio\n(Balanced risk)')

plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Risk Contribution: Equal Dollar Portfolio\n(Equities dominate risk!)" and "Risk Contribution: Risk Parity Portfolio\n(Balanced risk)". The chart highlights key patterns that inform the modelling approach.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Weight Shift:** To balance risk, the model significantly reduced the weight of Equities and Commodities (high risk) and increased the weight of T-Bills and Long Bonds (low risk).
2. **All Weather:** This approach is famous for being the core of Bridgewater's 'All Weather' fund. It performs more consistently across different economic cycles (inflation vs. deflation).
3. **Leverage Note:** In the real world, because Risk Parity portfolios have high weights in low-return assets (like T-Bills), managers often use leverage to boost the total return while keeping the risk balanced.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: compare risk-appetite portfolios ---
print("Operational demonstration — portfolios for different client profiles:\n")

cons = minimize(port_volatility, initial_weights, args=(df_returns,),
                method='SLSQP', bounds=bounds, constraints=constraints)
cons_ret, cons_vol, cons_sharpe = portfolio_stats(cons.x, df_returns)

print("Conservative (minimum volatility):")
for a, w in zip(assets, cons.x):
    if w > 0.01:
        print(f"  {a}: {w:.1%}")
print(f"  Return: {cons_ret:.2%}  Vol: {cons_vol:.2%}  Sharpe: {cons_sharpe:.2f}\n")

print("Aggressive (maximum Sharpe):")
for a, w in zip(assets, best_weights):
    if w > 0.01:
        print(f"  {a}: {w:.1%}")
print(f"  Return: {res_return:.2%}  Vol: {res_vol:.2%}  Sharpe: {res_sharpe:.2f}")

print("\nA wealth advisor would adjust these for client-specific constraints.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end wealth management and portfolio optimisation workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Optimisation models may suggest portfolios unsuitable for clients with different risk tolerances or time horizons.
- Model assumptions about return distributions may not hold during market crises.
- Fiduciary duty requires that model outputs support, not replace, professional judgement.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in wealth management and portfolio optimisation?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.